## TestModelOverClusters_ControlledForSampleSize

This script evaluates how predictive performance varies with sample size and number of clusters, while controlling for unequal image counts across clusters.

*Data*  
- Reads the expanded dataframe (one row per image) with cluster assignments for k=2..10 (output of script 5)
- Reads SES-WOA data (for use in model fitting)
- Loads the best XGBoost model from script 4

*Subsampling*  
For each value of k (2 to 10):
- The dataframe is grouped by the corresponding cluster label (scene_cluster_k).
- For each requested sample size N [500, 1000, 2000, ..., 20000]:
    - Up to N images are randomly sampled from each cluster independently.
    - If a cluster contains fewer than N images, all available images are used (i.e. the sample is capped).
    - The effective number of samples per cluster is recorded for later diagnostics.

This subsampling procedure is repeated 10 times using different random seeds to capture sampling variability.

*Feature construction*  
For each subsampled dataset:
- Images are grouped by LSOA.
- Within each LSOA and cluster, the median embedding is computed.
- These per-cluster median embeddings are merged into a single modelling table.
- The continuous SES-WOA score is joined to provide the prediction target.

*Model fitting*  
For each combination of k, subsample size N, random repeat, and individual cluster:
- If a cluster was capped (i.e. contains fewer than N images), model fitting for that cluster is skipped.
- LSOAs with missing embeddings are removed.
- A random train-test split (80/20) is applied.
- An XGBoost model is trained to predict the SES-WOA score using only the median embedding from the given cluster.
- R2, RMSE, and MAE are recorded.

### Dependencies
**Prerequisites:** Scripts 4 and 5 (needs best model and cluster-assigned pickle).

**Inputs:**
- `data/processed/one_row_per_image_cleaned_with_cluster_numbers.pkl` — from script 5
- `data/models/best_model.joblib` — from script 4 (cloned for per-cluster evaluation)
- CBS SES-WOA target (via `amsterdam_data.get_ses_woa()`)
- `clustering_functions.py` — imports `global_k`, `RANDOM_STATE`, `embedding_statistic`, `agg_funcs`

**Outputs:**
- `data/models/model_testing/cluster_subsample_results.pkl`
- Figure: model performance vs sample size across cluster counts

**Used by:** None (terminal analysis notebook)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import reduce

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from joblib import Parallel, delayed
import joblib


def is_missing_embedding(x):
    return isinstance(x, float) and np.isnan(x)

In [ ]:
from directory_filepaths import *
from clustering_functions import global_k, RANDOM_STATE, embedding_statistic, agg_funcs
from amsterdam_data import get_ses_woa

print(f"Using '{embedding_statistic}' embedding aggregation (from clustering_functions.py)")

In [ ]:
# Number of clusters
k = global_k
print(f"Number of clusters (defined in clustering_functions.py) = {global_k}")

# Target: continuous SES-WOA score
y_var = "ses_woa_score"

In [ ]:
# Specify the size of subsamples we wish to make
sample_sizes = [500,1000,2000,3000,4000,5000,6000,7000,10000,12500,15000,20000]

### Load best model from the model testing
Clone it so it doesn't have parameters fitted already

In [ ]:
best_model = joblib.load(os.path.join("../data/models/best_model.joblib"))['model']
model = clone(best_model)

### Get data

In [ ]:
# Load expanded dataframe with cluster assignments (output of script 5)
expanded_gdf = pd.read_pickle(os.path.join(data_dir, "one_row_per_image_cleaned_with_cluster_numbers.pkl"))
print(f"Loaded {len(expanded_gdf)} image rows")

In [ ]:
# for num in range(1,8):
#     print(num)
#     print(len(expanded_gdf[expanded_gdf['scene_cluster_7']==num]))

In [ ]:
# final_df = final_df.merge(lsoa_summary, on = "LSOA21CD")
# file_ending = f'kmeanscluster{k}_resampled1'
# final_df.to_pickle(data_dir + f"embedding_summaries/big_summary_df_{file_ending}.pkl")

### Read in SES-WOA data

In [ ]:
# SES-WOA target (the deprivation measure); keyed on buurtcode (JOIN_KEY)
ses = get_ses_woa()

# Test model performance 


In [ ]:
def sub_sample_data(df, cluster_col, n_per_cluster, random_state=None):
    """
    Sample up to n_per_cluster rows from each cluster.
    If a cluster has fewer rows, all are used (capped).
    """
    sampled = []
    for _, group in df.groupby(cluster_col):
        sampled.append(group.sample(n=min(len(group), n_per_cluster), random_state=random_state))
    return pd.concat(sampled).reset_index(drop=True)


def fit_models_for_k_sample(
    k, sample_size, repeat_id, expanded_gdf, ses, base_model, base_seed=RANDOM_STATE
):
    """
    For a given k and sample_size, subsample images, compute per-buurt embeddings
    (using embedding_statistic from clustering_functions.py) per cluster,
    and fit an XGBoost model for each cluster.
    """
    random_state = base_seed + repeat_id
    cluster_col = f"scene_cluster_{k}"
    categories = range(1, k + 1)

    # Cluster sizes in the full dataset
    cluster_sizes = expanded_gdf[cluster_col].value_counts()

    # Balanced subsample
    df = sub_sample_data(
        expanded_gdf, cluster_col,
        n_per_cluster=sample_size, random_state=random_state,
    )

    # Record actual sample counts and whether each cluster was capped
    actual_counts = df[cluster_col].value_counts().reindex(categories, fill_value=0).to_dict()
    sampling_status = {
        c: "full" if cluster_sizes.get(c, 0) >= sample_size else "capped"
        for c in categories
    }

    # Aggregate embedding per buurt, for each cluster
    # (uses the centrally-defined statistic from clustering_functions.py)
    embed_func = agg_funcs[embedding_statistic]
    per_cluster_dfs = [
        df[df[cluster_col] == cat]
        .groupby(JOIN_KEY)["embedding"]
        .apply(embed_func)
        .reset_index()
        .rename(columns={"embedding": f"{cat}_{embedding_statistic}"})
        for cat in categories
    ]

    final_df = reduce(
        lambda left, right: pd.merge(left, right, on=JOIN_KEY, how="outer"),
        per_cluster_dfs,
    )

    # Join SES-WOA target
    final_df = final_df.merge(ses[[JOIN_KEY, "ses_woa_score"]], on=JOIN_KEY, how="left")

    # Fit one model per cluster
    r2_results = {}
    mae_results = {}
    rmse_results = {}
    effective_n = {}

    for cluster_num in categories:
        # Skip capped clusters (not enough data for fair comparison)
        if sampling_status[cluster_num] == "capped":
            r2_results[cluster_num] = np.nan
            mae_results[cluster_num] = np.nan
            rmse_results[cluster_num] = np.nan
            effective_n[cluster_num] = actual_counts[cluster_num]
            continue

        col = f"{cluster_num}_{embedding_statistic}"
        df_cluster = final_df[[JOIN_KEY, col, y_var]].copy()
        df_cluster = df_cluster[~df_cluster[col].apply(is_missing_embedding)]

        effective_n[cluster_num] = len(df_cluster)

        if len(df_cluster) < 5:
            r2_results[cluster_num] = np.nan
            mae_results[cluster_num] = np.nan
            rmse_results[cluster_num] = np.nan
            continue

        X = np.stack(df_cluster[col].values)
        y = df_cluster[y_var].values
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=random_state,
        )

        # Clone the base model so each fit starts fresh
        this_model = clone(base_model)
        this_model.fit(X_train, y_train)
        y_pred = this_model.predict(X_test)

        r2_results[cluster_num] = r2_score(y_test, y_pred)
        rmse_results[cluster_num] = np.sqrt(mean_squared_error(y_test, y_pred))
        mae_results[cluster_num] = mean_absolute_error(y_test, y_pred)

    return {
        "k": k,
        "sample_size": sample_size,
        "repeat": repeat_id,
        "r2_results": r2_results,
        "rmse_results": rmse_results,
        "mae_results": mae_results,
        "actual_sample_counts": actual_counts,
        "effective_n_after_filtering": effective_n,
        "sampling_status": sampling_status,
    }


results_path = os.path.join("../data/models/model_testing", "cluster_subsample_results.pkl")

if os.path.exists(results_path):
    print(f"Results already exist at {results_path}, skipping computation. Delete the file to re-run.")
    outputs = pd.read_pickle(results_path)
else:
    # Only pass the columns workers actually need (saves memory per worker)
    cluster_cols = [f"scene_cluster_{k}" for k in range(2, 11)]
    slim_gdf = expanded_gdf[[JOIN_KEY, "embedding"] + cluster_cols].copy()

    n_repeats = 10
    n_jobs = 8
    k_values = range(2, 11)

    outputs = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(fit_models_for_k_sample)(
            k, sample_size, repeat_id, slim_gdf, ses, model,
        )
        for k in k_values
        for sample_size in sample_sizes
        for repeat_id in range(n_repeats)
    )

### Format outputs

Output structure

The script returns, for every experiment run:

 - the number of clusters k, the target sample size N, the repeat index, the effective sample counts per cluster, per-cluster model performance (R2)

Results are subsequently aggregated across repeats to compute:
- median R2  as a function of sample size
- variability due to random subsampling
- convergence behaviour of predictive performance with increasing data availability.

In [ ]:
# Save results (only needed after fresh computation; harmless to re-run)
os.makedirs(os.path.dirname(results_path), exist_ok=True)
pd.to_pickle(outputs, results_path)
print(f"Saved {len(outputs)} results to {results_path}")

In [ ]:
# Reload results from disk (useful if kernel was restarted after the cell above)
# outputs = pd.read_pickle(results_path)

In [ ]:
### Convert results to a tidy dataframe
rows = []

for run in outputs:
    k = run["k"]
    sample_size = run["sample_size"]
    repeat = run["repeat"]

    r2_dict   = run["r2_results"]
    rmse_dict = run["rmse_results"]
    mae_dict  = run["mae_results"]

    for cluster_num in r2_dict.keys():
        rows.append({"k": k, "sample_size": sample_size,
            "repeat": repeat, "cluster": cluster_num,
            "r2": r2_dict.get(cluster_num, np.nan),
            "rmse": rmse_dict.get(cluster_num, np.nan),
            "mae": mae_dict.get(cluster_num, np.nan)})

results_df = pd.DataFrame(rows)

In [ ]:
### Summarise results over samples
# Derive NRMSE from R²: NRMSE = sqrt(1 - R²), valid because R² = 1 - (RMSE/std(y))²
results_df["nrmse"] = np.sqrt(1 - results_df["r2"].clip(upper=1.0))

summary = (results_df.groupby(["k", "sample_size", "cluster"]).agg(
    mean_r2   = ("r2", "mean"),
    std_r2    = ("r2", "std"),

    mean_rmse = ("rmse", "mean"),
    std_rmse  = ("rmse", "std"),

    mean_mae  = ("mae", "mean"),
    std_mae   = ("mae", "std"),

    mean_nrmse = ("nrmse", "mean"),
    std_nrmse  = ("nrmse", "std"),

    n=("r2", "count")).reset_index()
           )

In [ ]:
summary

### Plot the results

**Note on metric choice:** Raw R² and RMSE are not directly comparable across clusters because 
each cluster's model is evaluated on a different subset of LSOAs (with different deprivation 
distributions). They *are* valid for tracking how a single cluster's performance changes with 
sample size.

For cross-cluster comparison we use **NRMSE** (Normalised RMSE = RMSE / std(y_test)), which is 
scale-free and comparable across subsets. It can be derived from R² as sqrt(1 − R²), so no 
recomputation is needed. Lower NRMSE is better, and a value of 1 corresponds to random performance
(same RMSE as predicting the mean).

In [ ]:
# Change this to "r2", "rmse", "mae", or "nrmse".
# NRMSE is recommended for cross-cluster comparison (see note above).
statistic_to_plot = "nrmse"

fig, axs = plt.subplots(ncols=3, nrows=3, figsize=(12, 10), sharex=True, sharey=True)
axs = axs.flatten()

for ax_num, k in enumerate(range(2, 11)):

    df_k = summary[summary["k"] == k]

    for cluster_num in sorted(df_k["cluster"].unique()):

        df_c = df_k[df_k["cluster"] == cluster_num]

        axs[ax_num].plot(
            df_c["sample_size"],
            df_c[f"mean_{statistic_to_plot}"],
            marker="o",
            label=f"Cluster {cluster_num}")
        
        axs[ax_num].fill_between(
        df_c["sample_size"],
        df_c[f"mean_{statistic_to_plot}"] - df_c[f"std_{statistic_to_plot}"],
        df_c[f"mean_{statistic_to_plot}"] + df_c[f"std_{statistic_to_plot}"],
        alpha=0.2)

    axs[ax_num].set_xlabel("Sample size")
    axs[ax_num].set_ylabel(f"Mean {statistic_to_plot.upper()}")
    axs[ax_num].set_title(f"{statistic_to_plot.upper()} vs sample size (k={k})")
    axs[ax_num].legend(fontsize=7)
    
fig.tight_layout()
fig.savefig(os.path.join(outputs_dir, "6-R2_vs_samplesize.pdf"), bbox_inches="tight")